# 04 · Revision analyses (npj Digital Medicine, reviewer comments)

Everything the reviewers asked for that needs numbers or figures, in one place.
Each section names the comment it answers, writes its outputs to
`results/revision/<metric>/<section>/` and is **resumable**: a finished fit is
loaded from disk instead of refitted (set `FORCE = True` to redo).

| Part | Section | Comments | Cost |
| :--- | :------ | :------- | :--- |
| 1 New fits | A1 Out-of-sample validation (+ Figs 2-4 on held-out users) | R1-M2, R3-M1 | heavy (one fit per fold) |
| | A2 Within- vs between-person temperature | R1-M6 | 2 fits |
| | A3 Country clustering, per-region fits | R1-M3 | 3 fits + nested fit on a capped sample + 1 per region |
| | A4 HRV lower bound | R1-M5 | light (daily data needed for the full answer) |
| | A5 Monthly thresholds, adherence | R1-m5 | light / 4 fits if valid-day counts exist |
| | A6 Untried combinations | R3-M2 | 6 fits |
| 2 Post hoc | B1 p-values | R3-M5 | light |
| | B2 RHR-HRV correlation | R1-m15 | light |
| | B3 Effect sizes in intuitive units | R1-m9 | light |
| | B4 Country residual vs obesity / PM2.5 (optional) | R1-m3 | light, external file needed |
| 3 Figures | C1 Fig 2 by hemisphere | R3-M6 | uses A1 |
| | C2 HRV fits on the ln scale, scale-free fit metrics | R3-M7 | uses A1 |
| | C3 Fig 1 with shared axes | R3-m1 | light |

Points that need an author decision are marked **REVIEW** (in the markdown
and as `REVIEW:` comments in `src/revision_lib.py`).

**Data the public schema does not have** (flagged where used):
valid-day counts per user-month (A5), daily values (A4 full answer, A5
refits from daily data, R2-3 weekday/weekend), device type (R3-M8; not
analysed).

For the long fits on the full data you can also run
`python src/run_revision.py --metric rhr --sections A1,A2` in a terminal
(same functions, same outputs, safe to interrupt and restart).

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import os
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath('../src'))
import file_io as fio
import publication_figures as pf
import revision_lib as rl

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

In [ ]:
# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
METRICS = ('rhr', 'hrv')
# results/ sub-folder of the 02_modelling run per metric (winner formula,
# feature stats, REML coefficients are read from there).
RUN_TAGS = {'rhr': 'rhr_rerun', 'hrv': 'hrv_rerun'}
# None = the BIC winner of that run. Give a formula string to override.
FORMULA_OVERRIDE = {'rhr': None, 'hrv': None}
SUBSAMPLE_FRAC = 1.0  # the paper's fits used everyone; <1 only for quick test runs
RANDOM_STATE = 42
FORCE = False  # True = refit even if results exist

# Which sections to run (the heavy ones can be switched off).
RUN = dict(A1=True, A2=True, A3=True, A4=True, A5=True, A6=False,  # A6 declined (R3-M2), kept optional
           B1=True, B2=True, B3=True, B4=True, C1=True, C2=True, C3=True)

# A1 out-of-sample validation: countries held out one at a time (ISO alpha-2),
# one per continent, matching the countries shown in Fig 4 where possible.
# The A1 cell prints each country's number of users; countries with fewer than
# MIN_USERS_PER_FOLD users are skipped. REVIEW: swap any country here.
HOLDOUT_COUNTRIES = (
    'GB',  # Europe (in Fig 4)
    'IN',  # Asia, tropical (in Fig 4; HRV underpredicted in summer)
    'CA',  # North America (the US is too large a share to hold out)
    'BR',  # South America, southern hemisphere (in Fig 4)
    'ZA',  # Africa, southern hemisphere (in Fig 4)
    'AU',  # Oceania, southern hemisphere (in Fig 4)
)
MIN_USERS_PER_FOLD = 200  # REVIEW: levels with fewer users are not held out
# Countries with fewer users than this (in the fitting sample) are pooled and
# held out together as one 'data-sparse countries' fold, to test the claim that
# the model can estimate trends where data are sparse. REVIEW: threshold
# (countries with fewer than 1,000 users in the cohort). None = skip.
SPARSE_COUNTRY_MAX_USERS = 1000
# Monthly means drawn / scored only for cells with at least this many rows.
MIN_ROWS_PER_CELL = 30
# A3: optional extra refit with Mid North users randomly reduced to this share
# of user-months. Off (None): the per-region refits already show robustness
# across regions, which is what R1-M3 asks for. Set e.g. 0.5 to run it.
REDUCE_MAX_SHARE = None  # A3: Mid North share of user-months after reduction
MIN_USERS_PER_REGION = 0  # fit every UN region; filter on n_users when reporting
# A5: CSV with columns id, month, n_valid_days per metric (valid-day counts
# exported from the internal pipeline; a valid day = daily value within
# 20-200 bpm / ms, the same bounds used to build the monthly means). None = threshold refits skipped.
VALID_DAYS_CSV = {'rhr': None, 'hrv': None}
# A4b (R1-M5): HRV lower-bound sensitivity, 20 ms (paper) vs HRV_LOW_BOUND.
HRV_LOW_BOUND = 10
# The HRV cohort re-extracted upstream with HRV_LOW_BOUND instead of 20 ms
# (same schema as the cohort CSV; new user-months come back in).
HRV_LOWBOUND_COHORT_CSV = None
# Best: from the SAME re-extraction, also the 20 ms version, so both bounds are
# compared on exactly the same people. If not given, the original 20 ms cohort
# and the 10 ms cohort are both restricted to users present in both.
HRV_20MS_SAME_EXTRACT_CSV = None
# Nightly HRV (columns user_id, date, value; NOT range-filtered): used for the
# what-the-bound-removes table, and as a fallback refit on the same user-months
# if no re-extracted cohort is given.
NIGHTLY_HRV_CSV = None
# B4: optional CSV with columns country (ISO alpha-2) + indicators.
EXTERNAL_INDICATORS_CSV = None

data_dir = fio.get_data_dir()
pf.setup_nature_style()
print('data:', data_dir)

In [ ]:
# ----------------------------------------------------------------------------
# Load and prepare (identical to 02_modelling: thinning, standardisation on the
# full cohort, all users (SUBSAMPLE_FRAC = 1.0), users with >= 2 months).
# ----------------------------------------------------------------------------
DATA, STATS, FORMULA, OUT = {}, {}, {}, {}
for metric in METRICS:
  df, st = rl.load_prepared(
      metric, data_dir, subsample_frac=SUBSAMPLE_FRAC,
      random_state=RANDOM_STATE)
  if VALID_DAYS_CSV.get(metric):
    rl.attach_valid_days(df, VALID_DAYS_CSV[metric])
  DATA[metric], STATS[metric] = df, st
  FORMULA[metric] = FORMULA_OVERRIDE[metric] or rl.winner_formula(
      metric, data_dir, RUN_TAGS[metric],
      fallback=rl.fallback_formula(metric))
  OUT[metric] = lambda *p, m=metric: rl.revision_dir(data_dir, m, *p)
  print(f'{metric.upper()}: {len(df):,} rows, {df["user_id_n"].nunique():,}'
        f' users\n  formula: {FORMULA[metric]}')

## Part 1 · New fits

### A1 · Out-of-sample validation (R1-M2, R3-M1)

The model is fitted on one set of participants and scored on participants it
has never seen:

* **User split**: 80% of users for fitting, 20% held out.
* **Leave one group out**: all users of one age × sex group, latitude band,
  UN region or country are held out in turn (users are assigned to their
  modal group, so no person is in both sets).
* **Data-sparse countries**: all countries with fewer than
  `SPARSE_COUNTRY_MAX_USERS` users are pooled and held out together. With the
  UN-region folds (e.g. all of Africa or South America unseen), this is the
  direct test of the "normative trends in data-sparse regions" claim; the
  single large countries test generalisation to unseen but well-sampled places.

For each held-out user, half of their months (at random) are used only to
estimate their personal intercept; both predictions are scored on the other
half:

* **marginal**: fixed effects only (a new person, nothing known about them);
* **conditional**: plus the intercept estimated from their other months.

Temperature is residualised on the training users only. HRV is scored on the
fitted ln scale and in ms. `agg_r` / `agg_nrmse` compare monthly mean
trajectories (what Figs 2-4 show); NRMSE is RMSE divided by the spread of the
observed monthly means, so it is comparable between RHR and HRV.

**How to read the marginal numbers.** Without the personal intercepts the
model cannot know an individual's baseline, so person-level marginal errors are
large everywhere, in-sample too (marginal R² is about 5% for RHR and 21% for
HRV). Judge held-out folds against the in-sample marginal numbers, not against
the conditional ones. The claim about normative trends is about monthly
population means, where personal intercepts average out: use the monthly-mean
columns. A held-out country or region can still sit above or below the
prediction as a whole (its users' intercepts need not average to zero), so the
monthly-mean error is split into a constant **level offset** and the **error
after removing that offset** (the seasonal shape), alongside the correlation
and the amplitude ratio.

**REVIEW**: hold-out countries (`HOLDOUT_COUNTRIES`, one per continent by default); 50/50 calibration split;
whether to report all folds or a summary in the supplement.

**Quick check before the long run:** users per country (the per-country table
R1-M3 asks for, saved as `A3_country/country_counts.csv`) and the counts for
the hold-out countries. No model fitting; takes seconds.

In [ ]:
for metric in METRICS:
  counts = rl.country_counts(DATA[metric])
  rl.write_csv(counts, os.path.join(OUT[metric]('A3_country'),
                                    'country_counts.csv'), index=True)
  print(f'{metric.upper()}: {len(counts)} countries; top 15 and hold-outs')
  display(counts.head(15))
  display(counts.reindex(list(HOLDOUT_COUNTRIES)))

In [ ]:
if RUN['A1']:
  OOS = {}
  for metric in METRICS:
    folds = rl.default_folds(DATA[metric], HOLDOUT_COUNTRIES,
                             MIN_USERS_PER_FOLD, SPARSE_COUNTRY_MAX_USERS)
    print(f'{metric.upper()}: {len(folds)} folds')
    OOS[metric] = rl.run_oos_validation(
        DATA[metric], FORMULA[metric], metric, OUT[metric]('A1_oos'),
        folds=folds, random_state=RANDOM_STATE, force=FORCE)
    tab = rl.oos_summary_table(OOS[metric], metric)
    rl.write_csv(tab, os.path.join(OUT[metric]('A1_oos'),
                                   'oos_summary_table.csv'))
    display(tab.round(3))

**Data-sparse countries, broken down.** The pooled data-sparse fold mixes
countries from both hemispheres and several climates, so its single pooled
monthly curve is not meaningful (opposite seasons cancel). Its predictions are
scored here within each latitude band, hemisphere and UN region instead; each
person is still predicted from their own location, month and temperature.

In [ ]:
if RUN['A1']:
  for metric in METRICS:
    fold = f'country_size={rl.SPARSE_LEVEL}'
    try:
      pred = rl.load_fold_predictions(OUT[metric]('A1_oos'), fold)
    except Exception:  # fold skipped (too few users) or not run yet
      print(f'{metric.upper()}: no data-sparse fold')
      continue
    parts = []
    for col in ('lat_band', 'hemisphere', 'un_region'):
      a = rl.group_agreement(pred, col, metric, log_scale=True,
                             min_rows=MIN_ROWS_PER_CELL)
      parts.append(a.rename(columns={col: 'group'}).assign(grouping=col))
    tab = pd.concat(parts, ignore_index=True)
    rl.write_csv(tab, os.path.join(OUT[metric]('A1_oos'),
                                   'sparse_countries_by_group.csv'))
    print(metric.upper())
    display(tab[['grouping', 'group', 'n_users', 'agg_r', 'agg_level_bias',
                 'agg_shape_rmse', 'amplitude_ratio']].round(3))

**Figs 2-4 redrawn on held-out users** (user split): global monthly means
(Fig 2), by latitude band (Fig 3) and for the countries with most held-out
users (Fig 4). Red = fixed-effect prediction, green dashed = with the
intercept estimated from the user's other months.

In [ ]:
if RUN['A1']:
  for metric in METRICS:
    pred = rl.load_fold_predictions(OUT[metric]('A1_oos'), 'user_split')
    fdir = OUT[metric]('A1_oos', 'figures')
    figs = {
        'fig2_heldout_global': rl.plot_monthly_trajectories(
            pred, metric, show_conditional=True, min_rows=MIN_ROWS_PER_CELL,
            title=f'{metric.upper()} held-out users'),
        'fig3_heldout_lat_band': rl.plot_monthly_trajectories(
            pred, metric, 'lat_band', show_conditional=True,
            min_rows=MIN_ROWS_PER_CELL),
        'fig4_heldout_countries': rl.plot_monthly_trajectories(
            pred, metric, 'country_2_alpha_cd',
            levels=list(pred['country_2_alpha_cd'].value_counts().index[:9]),
            show_conditional=True, min_rows=MIN_ROWS_PER_CELL),
    }
    for name, fig in figs.items():
      print(rl.save_fig(fig, fdir, name))
    plt.show()
    plt.close('all')

### A2 · Within- vs between-person temperature (R1-M6)

Each residualised temperature term is split into the person's own mean over
the year (between-person) and the month's deviation from it (within-person).
The winner is refitted with the within parts in place of the originals
(keeping the season interactions) plus the between parts as main effects.
The table puts the original, within and between coefficients side by side;
`p_diff` tests within = between.

Note for the reply: the current Fig 6 (`compute_within_user_sensitivity`)
already demeans both the target and the temperature residual per user, so the
**figure** is within-person; the reviewer's point applies to the **model
coefficients**. The curve below is the model-implied within-person effect,
which can be overlaid on Fig 6.

**REVIEW**: between parts as main effects only; squared within term =
(within deviation)²; the curve assumes mean, max and min temperature shift
together.

In [ ]:
if RUN['A2']:
  WB = {}
  for metric in METRICS:
    out = rl.run_within_between(DATA[metric], FORMULA[metric], metric,
                                OUT[metric]('A2_within_between'), force=FORCE)
    WB[metric] = out
    print(metric.upper(), '| BIC original:',
          round(out['fit']['original']['bic'], 1), '| within/between:',
          round(out['fit']['within_between']['bic'], 1))
    display(out['coefficients'].round(4))
    if 'coef_within' in out['coefficients']:
      curve = rl.within_temperature_curve(out['coefficients'], STATS[metric],
                                          metric)
      orig = rl.within_temperature_curve(out['coefficients'], STATS[metric],
                                         metric, column='coef_original')
      fig, ax = plt.subplots(figsize=(3.4, 2.4))
      ax.plot(curve['delta_c'], curve['effect'], label='within-person')
      ax.plot(orig['delta_c'], orig['effect'], '--', label='original model')
      ax.axhline(0, color='k', lw=0.5)
      ax.set_xlabel('Temperature deviation from own mean (°C)')
      ax.set_ylabel('Change (%)' if metric == 'hrv' else 'Change (bpm)')
      ax.legend(fontsize=7, frameon=False)
      rl.write_csv(curve, os.path.join(OUT[metric]('A2_within_between'),
                                       'within_curve.csv'))
      print(rl.save_fig(fig, OUT[metric]('A2_within_between'),
                        'within_curve'))
      plt.show()

### A3 · Geographic imbalance and country-level clustering (R1-M3)

Users are **not** assigned to a single country: everything uses the country
(and latitude band) of each user-month, so the few users who move between
countries simply contribute their months to each.

1. Users and user-months per country (supplementary table), and how many users
   appear in more than one country or latitude band during the year.
2. Share of the variance left after the fixed effects that lies between
   countries.
3. Coefficients of the winner side by side: (a) as fitted; (b) with country
   fixed effects (the country of each month), which absorb any shared
   country-level environment, socio-economic or behavioural level; (c) only if
   `REDUCE_MAX_SHARE` is set (off by default): refitted with Mid North users
   randomly reduced to that share of user-months.
4. The winner refitted separately on the user-months of each UN region.

**REVIEW**: country fixed effects also absorb
between-country latitude differences, so latitude terms will move in (b): the
seasonal and temperature terms are the ones to compare; minimum users per
region.

In [ ]:
if RUN['A3']:
  CTRY = {}
  for metric in METRICS:
    d = OUT[metric]('A3_country')
    CTRY[metric] = rl.run_country_robustness(
        DATA[metric], FORMULA[metric], metric, d,
        reduce_max_share=REDUCE_MAX_SHARE, force=FORCE)
    print(metric.upper())
    display(pd.Series(CTRY[metric]['variance']).to_frame('value'))
    display(CTRY[metric]['coefficients'].round(4))
    display(CTRY[metric]['counts'].head(20))
    path = os.path.join(d, 'coefficients_by_un_region.csv')
    if not fio.file_exists(path) or FORCE:
      byreg = rl.fit_by_group(DATA[metric], FORMULA[metric], 'un_region',
                              MIN_USERS_PER_REGION)
      rl.write_csv(byreg, path)
    byreg = rl.read_csv(path)
    if len(byreg):
      display(byreg.pivot(index='term', columns='un_region',
                          values='coef').round(4))

### A4 · HRV lower bound of 20 ms (R1-M5)

With the monthly data we can show how much of each sex × age group sits near
the bound and plot the distributions. The share of **daily** values removed
at 20 ms, and a refit with a lower bound, need the daily extract
(`rl.daily_bound_exclusions`, `rl.monthly_from_daily`; see the cell below).

**REVIEW / DATA NEEDED**: daily HRV values before filtering; the Fitbit
algorithm's own output floor (for the rationale in the reply).

In [ ]:
if RUN['A4'] and 'hrv' in DATA:
  raw_hrv = rl.add_groups(rl.load_raw('hrv', data_dir))
  low = rl.hrv_low_tail(raw_hrv)
  rl.write_csv(low, os.path.join(OUT['hrv']('A4_hrv_floor'), 'hrv_low_tail.csv'))
  display(low.round(2))
  fig = rl.plot_hrv_distribution_by_age(raw_hrv)
  print(rl.save_fig(fig, OUT['hrv']('A4_hrv_floor'), 'hrv_by_age'))
  plt.show()

In [ ]:
# With the daily extract (columns user_id, date, value; values NOT filtered):
#
# daily = pd.read_csv('daily_hrv.csv')
# info = raw_hrv.groupby('user_id')[['gender', 'age']].first().reset_index()
# display(rl.daily_bound_exclusions(daily, lo=20, hi=200, user_info=info))
# for lo in (None, 10, 15, 20):          # sensitivity of the lower bound
#   monthly = rl.monthly_from_daily(daily, lo=lo, hi=200)
#   ... merge `value` onto the cohort as avg_hrv and rerun A2 / the winner fit

### A4b · Refit with a lower HRV bound (R1-M5)

The paper drops nights below 20 ms before taking monthly means. Nightly values
run from about 6 to 150 ms (the device's own range) and peak at about 20 ms, so
the 20 ms bound removes roughly a quarter of nights, mostly from low-HRV
(older) users. Here the final HRV model is refitted with the lower bound
`HRV_LOW_BOUND` (default 10 ms) and compared with the 20 ms fit:

1. **What each bound removes** (needs `NIGHTLY_HRV_CSV`): share of nights below
   the bound, user-months dropped entirely (fewer than 4 valid nights), and the
   median / IQR of monthly HRV, overall and by sex x age band.
2. **Refit**: the final model on the 20 ms cohort vs the lower-bound cohort,
   same model, same centring and scaling, fitted by REML like the paper's
   tables. Coefficients side by side with difference and ratio, plus median
   HRV by age band.
   * Preferred input: `HRV_LOWBOUND_COHORT_CSV`, the cohort re-extracted
     upstream with the lower bound (new user-months come back in). A later
     re-extraction can lose users, so the comparison must be on the same
     people: best, also give `HRV_20MS_SAME_EXTRACT_CSV` (20 ms from the same
     re-extraction); otherwise both cohorts are restricted to the users present
     in both (counts saved to `common_users.json`).
   * Fallback: monthly means recomputed from `NIGHTLY_HRV_CSV` for the existing
     user-months only (understates the effect: user-months the 20 ms bound
     dropped cannot be added without their location / temperature rows).

Expect the age terms to move most; seasonal and temperature terms (relative,
within-person) less.

In [ ]:
if RUN['A4'] and 'hrv' in DATA:
  d = OUT['hrv']('A4b_hrv_lower_bound')
  lab = (f'20 ms', f'{HRV_LOW_BOUND:g} ms')
  if NIGHTLY_HRV_CSV:
    nightly = rl.read_csv(NIGHTLY_HRV_CSV)
    summ = rl.nightly_floor_summary(nightly, rl.load_raw('hrv', data_dir),
                                    bounds=(20.0, float(HRV_LOW_BOUND)))
    rl.write_csv(summ, os.path.join(d, 'what_each_bound_removes.csv'))
    display(summ.round(2))
  df_low, df_base = None, DATA['hrv']
  if HRV_LOWBOUND_COHORT_CSV:
    load = lambda path: rl.load_prepared(
        'hrv', data_dir, subsample_frac=SUBSAMPLE_FRAC,
        random_state=RANDOM_STATE, path=path,
        standardisation=STATS['hrv']['standardisation'])[0]
    df_low = load(HRV_LOWBOUND_COHORT_CSV)
    if HRV_20MS_SAME_EXTRACT_CSV:
      df_base = load(HRV_20MS_SAME_EXTRACT_CSV)
      print('Both bounds from the same re-extraction (same people)')
    else:
      df_base, df_low, info = rl.restrict_to_common_users(df_base, df_low)
      print('Original 20 ms vs re-extracted 10 ms, restricted to users in '
            'both:', info)
      rl._write_json(info, os.path.join(d, 'common_users.json'))
  elif NIGHTLY_HRV_CSV:
    df_low = rl.monthly_means_same_rows(nightly, DATA['hrv'],
                                        lo=float(HRV_LOW_BOUND))
    df_low['target_log'] = np.log(df_low[rl.TARGET_RAW['hrv']])
    print('Fallback: same user-months, monthly means recomputed')
  if df_low is None:
    print('A4b skipped: set HRV_LOWBOUND_COHORT_CSV or NIGHTLY_HRV_CSV')
  else:
    fl = rl.floor_sensitivity_refit(df_base, df_low, FORMULA['hrv'], d,
                                    labels=lab, force=FORCE)
    display(pd.DataFrame(fl['fit']))
    display(fl['coefficients'].round(4))
    display(fl['medians'].round(1))

### A5 · Monthly thresholds and adherence (R1-m5)

Adherence by calendar month and hemisphere: active users per month (always)
and valid days per user-month (only if a valid-day count is attached). With
valid-day counts, the winner is refitted keeping user-months with ≥ 7, 14 and
20 valid days.

**DATA NEEDED**: valid-day counts per user-month are not in the public
schema; set `VALID_DAYS_CSV` (columns `id, month, n_valid_days`).

In [ ]:
if RUN['A5']:
  for metric in METRICS:
    d = OUT[metric]('A5_thresholds')
    adh = rl.adherence_by_month(DATA[metric])
    rl.write_csv(adh, os.path.join(d, 'adherence_by_month.csv'))
    display(adh.round(2))
    fig, ax = plt.subplots(figsize=(3.4, 2.4))
    ycol = ('mean_valid_days' if 'mean_valid_days' in adh
            else 'share_of_users_%')
    for hemi, g in adh.groupby('hemisphere'):
      ax.plot(g['month'], g[ycol], 'o-', ms=3, label=hemi)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(rl.MONTH_TICKS)
    ax.set_ylabel(ycol)
    ax.legend(fontsize=7, frameon=False)
    ax.set_title(metric.upper(), fontsize=9)
    print(rl.save_fig(fig, d, 'adherence_by_month'))
    plt.show()
    if rl.VALID_DAYS_COL in DATA[metric]:
      thr = rl.run_threshold_sensitivity(DATA[metric], FORMULA[metric],
                                         metric, d, force=FORCE)
      display(thr.pivot(index='term', columns='min_valid_days',
                        values='coef').round(4))
    else:
      print(f'{metric.upper()}: no valid-day counts -> threshold refits skipped')

### A6 · Combinations the stage-wise selection did not try (R3-M2)

The winner plus one extra interaction at a time (age × temperature,
sex × temperature, sex × season, age × season, latitude × temperature), all
fitted by ML on the same rows; `delta_BIC_vs_winner` < 0 means the extra term
is preferred.

**REVIEW**: candidate list; whether to also rerun the competition with the
stage order reversed (`stage_competition.run_competition` with a custom
`StageSpec`).

In [ ]:
if RUN['A6']:
  for metric in METRICS:
    t = rl.run_extra_models(DATA[metric], FORMULA[metric], metric,
                            OUT[metric]('A6_extra_models'), force=FORCE)
    display(t[['candidate', 'bic', 'delta_BIC_vs_winner', 'marginal_R2',
               'conditional_R2']].round(4))

## Part 2 · Post hoc

### B1 · p-values for the group comparisons (R3-M5)

Tests on each user's mean value (RHR in bpm, HRV in ms):

* sex (men vs women) and hemisphere (northern vs southern);
* age: one Kruskal-Wallis test across all age bands, then each band vs the next
  one up (where along the lifespan the change happens);
* latitude: one Kruskal-Wallis test across all bands, tropics vs Far North /
  polar (the reviewer's comparison), tropics vs each other band (the "tropical
  RHR is highest" claim) and Mid North vs Far North (the HRV range in Results
  2.1.2).

Pairwise rows give the difference with its 95% CI, Cohen's d, Welch's t-test
and Mann-Whitney U; Kruskal-Wallis rows give H, p and epsilon² (share of rank
variance explained). Model terms already have p-values in the coefficient
tables. With this N, all p-values will be tiny; the reply can say so and point
to effect sizes.

**REVIEW**: keep only the rows that match a claim in the text.

In [ ]:
if RUN['B1']:
  tests = pd.concat([rl.group_comparisons(DATA[m], m) for m in METRICS])
  rl.write_csv(tests, os.path.join(OUT[METRICS[0]]('B1_tests'),
                                   'group_tests_all_metrics.csv'))
  display(tests.round(4))

### B2 · RHR-HRV correlation (R1-m15)

Users in both cohorts, matched by month: between-person correlation of user
means and within-person repeated-measures correlation.

**REVIEW**: user ids must match across the two files.

In [ ]:
if RUN['B2'] and set(METRICS) >= {'rhr', 'hrv'}:
  corr = rl.rhr_hrv_correlation(rl.load_raw('rhr', data_dir),
                                rl.load_raw('hrv', data_dir))
  display(pd.Series(corr))
  rl._write_json(corr, os.path.join(OUT['hrv']('B2_rhr_hrv'),
                                    'rhr_hrv_correlation.json'))

### B3 · Effect sizes in intuitive units (R1-m9)

Differences of the fixed-effect prediction between synthetic profiles that
differ in one variable, averaged over the year. No refitting, so it is instant.
`effect_sizes.csv` has the headline contrasts; `effect_size_grid.csv` has many
scenarios (age steps by sex and latitude, sex at each age, temperature
-5/+1/+5/+10 degC per month and per year at 11 latitudes, 10 deg further from
the equator, seasonal swing with peak and trough months by latitude), plotted
in `effect_size_grid.png`. Uses the paper's REML
coefficients and feature statistics from the 02_modelling run when present
(otherwise a REML refit here).

**REVIEW**: reference profile (age 40, 45° N) and the list of contrasts.

In [ ]:
if RUN['B3']:
  for metric in METRICS:
    fe = rl.winner_coefficients(metric, data_dir, RUN_TAGS[metric])
    st = rl.run_stats_from_run(metric, data_dir, RUN_TAGS[metric])
    if fe is None or st is None or FORMULA_OVERRIDE[metric]:
      print(f'{metric.upper()}: run coefficients not found -> REML refit')
      fe = rl.fit_lme(FORMULA[metric], DATA[metric], reml=True).fe_params
      st = STATS[metric]
    d = OUT[metric]('B3_effect_sizes')
    es = rl.effect_sizes(fe, st, metric)
    rl.write_csv(es, os.path.join(d, 'effect_sizes.csv'))
    display(es.round(2))
    grid = rl.effect_size_grid(fe, st, metric)
    rl.write_csv(grid, os.path.join(d, 'effect_size_grid.csv'))
    display(grid[grid['month'] == 0].pivot_table(
        index=['kind', 'from', 'to'], columns='latitude', values='effect',
        aggfunc='first').round(2))
    print(rl.save_fig(rl.plot_effect_grid(grid, metric), d, 'effect_size_grid'))
    plt.show()

### B4 · Country residuals vs obesity and air pollution (R1-m3, optional)

Only if the text keeps the obesity / pollution explanation as more than a
hypothesis. Needs a CSV with `country` (ISO alpha-2) and indicator columns.

In [ ]:
if RUN['B4'] and EXTERNAL_INDICATORS_CSV:
  for metric in METRICS:
    res = rl.fit_lme(FORMULA[metric], DATA[metric])
    cr = rl.country_residual_correlation(DATA[metric], res, metric,
                                         EXTERNAL_INDICATORS_CSV)
    rl.write_csv(cr, os.path.join(OUT[metric]('B4_country_residuals'),
                                  'country_residual_correlation.csv'))
    display(cr.round(3))
else:
  print('B4 skipped (no EXTERNAL_INDICATORS_CSV)')

## Part 3 · Figures

### C1 · Fig 2 by hemisphere (R3-M6)

Same as Fig 2, on held-out users, separately for the northern and southern
hemispheres (opposite seasons; the pooled curve is dominated by the north).

In [ ]:
if RUN['C1']:
  for metric in METRICS:
    pred = rl.load_fold_predictions(OUT[metric]('A1_oos'), 'user_split')
    fig = rl.plot_monthly_trajectories(
        pred, metric, 'hemisphere', levels=['Northern', 'Southern'],
        show_conditional=True, ncols=2, min_rows=MIN_ROWS_PER_CELL)
    print(rl.save_fig(fig, OUT[metric]('C_figures'), 'fig2_by_hemisphere'))
    plt.show()

### C2 · HRV fits on the ln scale and scale-free fit metrics (R3-M7)

Figs 3-4 for HRV redrawn on ln HRV (the scale the model is fitted on), plus
per-group agreement of monthly trajectories for both metrics: correlation,
NRMSE and amplitude ratio (predicted / observed peak-to-trough), which can be
compared between RHR and HRV.

In [ ]:
if RUN['C2']:
  agree = []
  for metric in METRICS:
    pred = rl.load_fold_predictions(OUT[metric]('A1_oos'), 'user_split')
    for col in ('lat_band', 'country_2_alpha_cd'):
      a = rl.group_agreement(pred, col, metric, log_scale=True,
                             min_rows=MIN_ROWS_PER_CELL)
      a.insert(0, 'metric', metric)
      agree.append(a.rename(columns={col: 'group'}).assign(grouping=col))
    if metric == 'hrv':
      for col, name in (('lat_band', 'fig3_hrv_ln'),
                        ('country_2_alpha_cd', 'fig4_hrv_ln')):
        levels = (None if col == 'lat_band' else
                  list(pred[col].value_counts().index[:9]))
        fig = rl.plot_monthly_trajectories(pred, 'hrv', col, levels=levels,
                                           log_scale=True,
                                           min_rows=MIN_ROWS_PER_CELL)
        print(rl.save_fig(fig, OUT['hrv']('C_figures'), name))
      plt.show()
  agree = pd.concat(agree, ignore_index=True)
  rl.write_csv(agree, os.path.join(OUT[METRICS[0]]('C_figures'),
                                   'group_agreement_rhr_vs_hrv.csv'))
  display(agree.groupby(['metric', 'grouping'])[
      ['agg_r', 'agg_nrmse', 'amplitude_ratio']].median().round(3))

### C3 · Fig 1 with shared axes (R3-m1)

Fig 1 regenerated with the same y-axis for women and men (A/B, C/D) and the
same age axis across A-D.

**REVIEW**: the permanent fix is to change the hard-coded `y_lim` values in
`publication_figures.generate_figure_1`; this cell shows the result first.

In [ ]:
if RUN['C3'] and set(METRICS) >= {'rhr', 'hrv'}:
  fig, paths = rl.figure_1_shared_axes(
      rl.load_raw('rhr', data_dir), rl.load_raw('hrv', data_dir),
      output_dir=OUT['rhr']('C_figures'))
  print(paths)
  plt.show()